# Deep Q-Learning for Wumpus World

This notebook trains and evaluates the simple `DeepWumpusAgent`.

In [ ]:
import gymnasium as gym
import numpy as np
from matplotlib import pyplot as plt
from tqdm import tqdm

from deep_q_agent import DeepWumpusAgent
from environment import WumpusEnv

In [ ]:
learning_rate = 0.001
n_episodes = 10_000
start_epsilon = 1.0
epsilon_decay = start_epsilon / (n_episodes / 2)
final_epsilon = 0.01
num_training_maps = 100  # Fixed pool enables train/test comparison
training_seed = 42
test_seed = 12345

env = gym.wrappers.RecordEpisodeStatistics(
    WumpusEnv(num_training_maps=num_training_maps, seed=training_seed),
    buffer_length=n_episodes,
)
agent = DeepWumpusAgent(
    env=env,
    learning_rate=learning_rate,
    initial_epsilon=start_epsilon,
    epsilon_decay=epsilon_decay,
    final_epsilon=final_epsilon,
)

In [ ]:
for episode in tqdm(range(n_episodes)):
    obs, info = env.reset()
    done = False

    while not done:
        action = agent.get_action(obs)
        next_obs, reward, terminated, truncated, info = env.step(action)

        agent.update(obs, action, reward, terminated, next_obs)

        obs = next_obs
        done = terminated or truncated

    agent.decay_epsilon()

In [ ]:
def moving_average(values, window=100):
    values = np.asarray(values, dtype=float).flatten()
    if len(values) < window:
        return values
    return np.convolve(values, np.ones(window) / window, mode="valid")


fig, axs = plt.subplots(ncols=3, figsize=(12, 4))

axs[0].set_title("Episode rewards")
axs[0].plot(moving_average(env.return_queue))
axs[0].set_xlabel("Episode")

axs[1].set_title("Episode lengths")
axs[1].plot(moving_average(env.length_queue))
axs[1].set_xlabel("Episode")

axs[2].set_title("Training loss")
axs[2].plot(moving_average(agent.training_error))
axs[2].set_xlabel("Training step")

plt.tight_layout()
plt.show()

In [ ]:
def evaluate_on_maps(agent, test_env, number_of_maps, label):
    old_epsilon = agent.epsilon
    agent.epsilon = 0.0  # Evaluation without random exploration

    rewards = []
    episode_lengths = []
    wins = 0
    deaths = 0
    timeouts = 0

    for map_id in range(number_of_maps):
        obs, info = test_env.reset(options={"map_id": map_id})
        done = False
        episode_reward = 0

        while not done:
            action = agent.get_action(obs)
            obs, reward, terminated, truncated, info = test_env.step(action)
            episode_reward += reward
            done = terminated or truncated

        rewards.append(episode_reward)
        episode_lengths.append(info["steps"])

        if terminated and reward == test_env.unwrapped.goal_reward:
            wins += 1
        elif terminated:
            deaths += 1
        else:
            timeouts += 1

    agent.epsilon = old_epsilon

    print(label)
    print(f"Win rate:       {wins / number_of_maps:.1%}")
    print(f"Death rate:     {deaths / number_of_maps:.1%}")
    print(f"Timeout rate:   {timeouts / number_of_maps:.1%}")
    print(f"Average reward: {np.mean(rewards):.2f}")
    print(f"Average steps:  {np.mean(episode_lengths):.2f}")


# Common benchmark used by all three agents.
from evaluation import evaluate_train_test

results = evaluate_train_test(
    agent,
    training_env=env,
    num_test_maps=100,
    test_seed=test_seed,
    label="Deep Q-Learning",
)